<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml06_regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

농어(Perch)의 무게 예측

*   농어의 모든 특성(길이, 대각선, 키, 두께)을 사용해서 무게 예측
    *   Weight ~ Length + Diagonal + Height + Width
*   KNN, Linear Regression 비교
*   특성들의 다차항을 포함하는 회귀
*   규제(Regularization)

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# Fish 데이터셋 준비

In [2]:
file_path = 'https://bit.ly/fish_csv_data'

In [3]:
fish = pd.read_csv(file_path)

In [4]:
fish.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    object 
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), object(1)
memory usage: 7.6+ KB


In [5]:
fish.head()

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [6]:
perch = fish[fish.Species == 'Perch']

In [7]:
perch.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, 72 to 127
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   56 non-null     object 
 1   Weight    56 non-null     float64
 2   Length    56 non-null     float64
 3   Diagonal  56 non-null     float64
 4   Height    56 non-null     float64
 5   Width     56 non-null     float64
dtypes: float64(5), object(1)
memory usage: 3.1+ KB


In [8]:
perch.head()

,Species,Weight,Length,Diagonal,Height,Width
72,Perch,5.9,8.4,8.8,2.1120,1.4080
73,Perch,32.0,13.7,14.7,3.5280,1.9992
74,Perch,40.0,15.0,16.0,3.8240,2.4320
75,Perch,51.5,16.2,17.2,4.5924,2.6316
76,Perch,70.0,17.4,18.5,4.5880,2.9415


In [9]:
perch.columns[2:]

Index(['Length', 'Diagonal', 'Height', 'Width'], dtype='object')

In [10]:
# 특성 배열(독립변수들) - 2차원 배열
x = perch[perch.columns[2:]].values
x.shape  #> (56, 4) = (n_samples, n_features)

(56, 4)

In [11]:
# 타겟 배열(종속변수) - 1차원 배열
y = perch.Weight.values
y.shape

(56,)

# 훈련 셋 vs 테스트 셋 나누기

In [12]:
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.25,
                                                    random_state=42)

In [13]:
x_train[:5]

array([[19.6   , 20.8   ,  5.1376,  3.0368],
       [22.    , 23.5   ,  5.875 ,  3.525 ],
       [18.7   , 19.4   ,  5.1992,  3.1234],
       [17.4   , 18.5   ,  4.588 ,  2.9415],
       [36.    , 38.3   , 10.6091,  6.7408]])

# 1차항들만 포함하는 회귀

## KNN

In [14]:
knn = KNeighborsRegressor()  # KNN 회귀 모델 생성

In [15]:
knn.fit(X=x_train, y=y_train)  # ML 모델 훈련

KNeighborsRegressor()

In [16]:
train_pred = knn.predict(X=x_train)  # 훈련 셋 (무게) 예측값
print(train_pred)

[  87.6  123.    79.6   70.6  723.   183.4  847.   847.  1020.   123.
   95.   123.   174.   248.  1043.   847.   174.   122.   248.   847.
  582.   224.   723.    60.   142.    60.   685.   694.2  248.   167.
  847.   122.   139.   123.  1020.   136.    79.6  685.   123.   193.
 1043.   659. ]


In [17]:
print(y_train)

[  85.  135.   78.   70.  700.  180.  850.  820. 1000.  120.   85.  130.
  225.  260. 1100.  900.  145.  115.  265. 1015.  514.  218.  685.   32.
  145.   40.  690.  840.  300.  170.  650.  110.  150.  110. 1000.  150.
   80.  700.  120.  197. 1100.  556.]


In [18]:
mean_squared_error(y_true=y_train, y_pred=train_pred)  # 훈련 셋 MSE

2986.5723809523806

In [19]:
knn.score(X=x_train, y=y_train)  # 훈련 셋 R2 score

0.97579760182756

In [20]:
test_pred = knn.predict(X=x_test)  # 테스트 셋 예측값

In [21]:
mean_squared_error(y_true=y_test, y_pred=test_pred)  # 테스트 셋 MSE

837.3100000000001

In [22]:
knn.score(X=x_test, y=y_test)  # 테스트 셋 R2 score

0.9916579819676246

KNN 회귀 모델은 테스트 셋에서 더 좋은 점수를 보여주고 있음 -> 과소적합(under-fitting)

## LinearRegression

*   Weight ~ Length + Diagonal + Height + Width
*   $ 무게 예측값 = w_0 + 길이 \times w_1 + 대각선 \times w_2 + 키 \times w_3 + 두께 \times w_4 $

In [23]:
lin_reg = LinearRegression()  # 선형 회귀 모델 생성

In [24]:
lin_reg.fit(X=x_train, y=y_train)  # 선형 회귀 모델 훈련

LinearRegression()

In [25]:
lin_reg.coef_  #> 훈련이 끝난 선형 회귀 모델에서 찾은 선형회귀식의 계수들

array([-40.18338554,  47.80681727,  67.34086612,  35.34904264])

In [26]:
lin_reg.intercept_  #> 훈련이 끝난 선형 회귀 모델에서 찾은 선형회귀식의 상수항(절편)

np.float64(-610.0275364260515)

In [28]:
train_pred = lin_reg.predict(X=x_train)  # 훈련 셋 (무게) 예측값

In [29]:
train_pred

array([  50.07831254,  149.63115115,   26.52323981,  -11.85322276,
        727.07849472,  216.11818851,  859.35210445,  894.24144157,
        883.76216601,  133.80604761,   30.46174313,  165.45625469,
        267.36647321,  302.42993565,  942.06583292,  859.73196835,
        209.15316045,  137.18128947,  294.64533152,  907.16858502,
        585.54863062,  292.8893912 ,  763.11655759, -149.53132283,
        163.94525857, -104.38889956,  718.95576629,  815.95759166,
        350.34538816,  195.07245372,  764.17125484,  130.77848264,
        116.61555757,  142.50754589,  959.21205119,  218.69399647,
         79.52715018,  737.86169572,  161.30274218,  243.72987423,
        939.22223984,  665.0680958 ])

In [30]:
# 행렬 곱셈을 사용한 선형회귀식: y = X @ w + w0
x_train @ lin_reg.coef_ + lin_reg.intercept_

array([  50.07831254,  149.63115115,   26.52323981,  -11.85322276,
        727.07849472,  216.11818851,  859.35210445,  894.24144157,
        883.76216601,  133.80604761,   30.46174313,  165.45625469,
        267.36647321,  302.42993565,  942.06583292,  859.73196835,
        209.15316045,  137.18128947,  294.64533152,  907.16858502,
        585.54863062,  292.8893912 ,  763.11655759, -149.53132283,
        163.94525857, -104.38889956,  718.95576629,  815.95759166,
        350.34538816,  195.07245372,  764.17125484,  130.77848264,
        116.61555757,  142.50754589,  959.21205119,  218.69399647,
         79.52715018,  737.86169572,  161.30274218,  243.72987423,
        939.22223984,  665.0680958 ])

In [31]:
mean_squared_error(y_true=y_train, y_pred=train_pred)  # train MSE

5340.176566753986

In [32]:
lin_reg.score(X=x_train, y=y_train)  # train R2 score

0.9567246116638569

In [33]:
test_pred = lin_reg.predict(X=x_test)  # 테스트 셋 (무게) 예측값

In [34]:
mean_squared_error(y_true=y_test, y_pred=test_pred)  # test MSE

12140.410523504848

In [35]:
lin_reg.score(X=x_test, y=y_test)  # test R2 score

0.8790465615990273

1차항들만 포함하는 선형 회귀 모델은 훈련 셋의 점수가 테스트 셋에서의 점수보다 훨씬 좋음

--> 과대적합(over-fitting)

# 2차항들을 포함하는 회귀

## KNN

In [37]:
poly = PolynomialFeatures(include_bias=True)

`PolynomialFeatures()` 생성자 파라미터들:

*   `degree`: 기본값 2. 몇 차항을 만들 지를 결정.
*   `interaction_only`: 기본값이 False.
    *   `interaction_only=False`: $ {x_0}^2, {x_1}^2, {x_2}^2, ..., x_0 x_1, x_0 x_2, ...  $
    *   `interaction_only=True`: $ x_0 x_1, x_0 x_2, ... $
*   `include_bias`: 기본값이 True.
    *   `include_bias=True`: 상수항(1) 컬럼이 포함.
    *   `include_bias=False`: 상수항(1) 컬럼이 포함되지 않음.
